# Computer Vision — โจทย์แบบ "หาวัตถุในรูป (กล่อง)" Object Detection

รูป 1 รูป -> หากล่อง (bounding box) + ป้ายของวัตถุ (เช่น หาบ้าน/รถ/ป้ายทะเบียนในรูป)

วิธีในโน้ตบุ๊กนี้: `YOLOv8` (ไลบรารี ultralytics) เป็นวิธีที่ง่ายและแรงสุดสำหรับมือใหม่
สั่งเทรน 2-3 บรรทัดจบ


## เมื่อไหร่ใช้โน้ตบุ๊กนี้

ใช้เมื่อ: โจทย์ให้ทำนาย "ตำแหน่งกล่อง + ชนิดวัตถุ" metric มักเป็น `mAP`
ถ้าโจทย์แค่ "รูปนี้คือคลาสอะไร" (ไม่มีกล่อง) -> กลับไป `image_classification.ipynb` (ง่ายกว่ามาก)

หมายเหตุมือใหม่: detection ยากกว่า classification เพราะข้อมูลต้องมี label เป็นกล่อง (รูปแบบ YOLO)
ถ้าเวลาน้อยและโจทย์เป็น classification ได้ ให้เลือก classification ก่อน

## ขั้นที่ 1 — ติดตั้ง

In [ ]:
!pip -q install ultralytics pandas pyyaml

## ขั้นที่ 2 — เอาข้อมูลเข้า (เลือก 1 ใน 3 วิธี) + เช็ค GPU

เซลล์ล่างรองรับ 3 วิธี แก้แค่ตัวแปรบนสุดให้ตรงกับวิธีที่ใช้:

วิธี A (แนะนำ) ดาวน์โหลดจาก Kaggle อัตโนมัติ: ต้องมี `kaggle.json` (kaggle.com -> รูปโปรไฟล์ -> Settings -> Account -> Create New Token)
- บน `Kaggle Notebook`: ข้อมูลต่อไว้ที่ `/kaggle/input/...` แล้ว ไม่ต้องใส่ token
- บน `Colab/เครื่อง`: ใส่ `KAGGLE_USERNAME` + `KAGGLE_KEY`

วิธี B โหลดข้อมูลมาเครื่องตัวเองแล้วอัปขึ้น Colab เอง: ลากไฟล์ (เช่น `data.zip`) ไปวางในแถบ Files ซ้ายมือของ Colab
แล้วตั้ง `DATA_DIR = "/content"` (หรือโฟลเดอร์ที่วาง) -> เซลล์จะแตก zip ให้เอง ไม่ต้องใช้ token

วิธี C ต่อ Google Drive: รัน `from google.colab import drive; drive.mount('/content/drive')` ก่อน แล้วตั้ง `DATA_DIR = "/content/drive/MyDrive/โฟลเดอร์ของคุณ"`

เซลล์นี้เช็ค GPU ให้ด้วย ถ้างานเป็น deep learning (รูป/ข้อความ/สัญญาณดิบ) ควรเปิด GPU: เมนู `Runtime > Change runtime type > T4 GPU`

In [ ]:
import os, glob, subprocess

# ----- วิธี A: ดาวน์โหลดจาก Kaggle -----
COMP = "ใส่-slug-ของ-competition-detection-ตรงนี้"      # << แก้: slug ท้าย URL เช่น kaggle.com/competitions/titanic -> "titanic"  (ใช้เฉพาะวิธี A)
KAGGLE_USERNAME = ""   # << แก้ (วิธี A, เฉพาะ Colab/เครื่อง) เช่น "peetwan1"  (บน Kaggle Notebook เว้นว่างได้)
KAGGLE_KEY      = ""   # << แก้ (วิธี A, เฉพาะ Colab/เครื่อง) เช่น "0a1b2c3d4e5f..." (คีย์ยาว ~32 ตัวจาก kaggle.json)
# ----- วิธี B/C: อัปโหลดเอง / ต่อ Google Drive -----
DATA_DIR = ""          # << แก้: ถ้าอัปโหลดข้อมูลเอง ใส่ path โฟลเดอร์ เช่น "/content" (วิธี B) หรือ "/content/drive/MyDrive/data" (วิธี C) ; ใช้ Kaggle (วิธี A) ปล่อยว่าง

# เช็ค GPU (ถ้าไม่มี + งานเป็น deep learning -> เปิดที่เมนู Runtime > Change runtime type > T4 GPU)
try:
    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "ไม่มี (งาน deep learning จะช้า; งานตาราง/ตัดคำ ไม่ต้องใช้ GPU ก็ได้)")
except Exception:
    pass

def get_data(comp):
    # วิธี B/C: ผู้ใช้ตั้ง DATA_DIR เอง (อัปโหลด/ต่อ Drive) -- แตก zip ในโฟลเดอร์นั้นให้ด้วยถ้ามี
    if DATA_DIR and os.path.isdir(DATA_DIR):
        for z in glob.glob(os.path.join(DATA_DIR, "*.zip")):
            subprocess.run(["unzip", "-o", "-q", z, "-d", DATA_DIR])
        print("ใช้ข้อมูลจากโฟลเดอร์ที่ตั้งเอง:", DATA_DIR); return DATA_DIR
    # บน Kaggle Notebook ข้อมูลต่อไว้แล้ว
    kpath = f"/kaggle/input/{comp}"
    if os.path.isdir(kpath):
        print("อยู่บน Kaggle ข้อมูลอยู่ที่", kpath); return kpath
    # วิธี A: ดาวน์โหลดด้วย Kaggle API
    if KAGGLE_USERNAME and KAGGLE_KEY:
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        open(os.path.expanduser("~/.kaggle/kaggle.json"), "w").write(
            '{"username":"%s","key":"%s"}' % (KAGGLE_USERNAME, KAGGLE_KEY))
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    os.makedirs("data", exist_ok=True)
    subprocess.run(["kaggle","competitions","download","-c",comp,"-p","data"], check=True)
    for z in glob.glob("data/*.zip"):
        subprocess.run(["unzip","-o","-q",z,"-d","data"], check=True)
    return "data"

DATA_DIR = get_data(COMP)
print("\nไฟล์ที่มี (ดูชื่อไฟล์/โฟลเดอร์ แล้วแก้ในเซลล์ถัดไปให้ตรง):")
for p in sorted(glob.glob(os.path.join(DATA_DIR, "**"), recursive=True))[:40]:
    print("  ", p)

## ขั้นที่ 3 — เตรียมข้อมูลแบบ YOLO

YOLO ต้องการ: โฟลเดอร์รูป + ไฟล์ label `.txt` (คลาส x_center y_center w h เป็นค่า 0-1) ต่อรูป 1 ไฟล์
และไฟล์ `data.yaml` ที่บอก path รูป train/val + ชื่อคลาส

สำคัญสำหรับมือใหม่: ถ้า competition ให้ข้อมูลเป็นรูปแบบอื่น (COCO json / csv ของกล่อง) ต้องแปลงเป็นรูปแบบ YOLO ก่อน
ถ้าตอนเทรนขึ้น error `No labels found in ...` แปลว่ายังไม่ได้แปลงข้อมูลเป็นรูปแบบ YOLO (ต้องสร้างไฟล์ .txt ต่อรูปก่อน)

In [ ]:
import yaml, os
# << แก้ path เหล่านี้ให้ตรงกับข้อมูลจริง (โฟลเดอร์ที่มีรูป + label .txt)
data_cfg = {
    "path": os.path.abspath(DATA_DIR),   # โฟลเดอร์หลัก (เขียน path สั้น ๆ ต่อจากนี้ในบรรทัดล่าง)
    "train": "images/train",             # << แก้: โฟลเดอร์รูป train เช่น "train/images" หรือ "dataset/images/train"
    "val":   "images/val",               # << แก้: โฟลเดอร์รูป val เช่น "valid/images" (ไม่มี val ให้แบ่งจาก train เองก่อน)
    "names": {0: "object"},              # << แก้: ชื่อคลาส เช่น {0:"house",1:"car"} (เลขต้องตรงกับเลขคลาสในไฟล์ label .txt)
}
with open("data.yaml","w") as f: yaml.safe_dump(data_cfg, f, allow_unicode=True)
print(open("data.yaml").read())

## ขั้นที่ 4 — เทรน YOLO (ง่ายสุด)

`yolo11n` = เล็ก/เร็ว, `yolo11s/m` = ใหญ่/แม่นขึ้น (รุ่นใหม่กว่า yolov8) ปรับ `epochs`/`imgsz` ตามเวลา/GPU
เคล็ดลับ: ลอง `epochs=10` ก่อนให้รันผ่าน แล้วค่อยเพิ่ม (epochs สูงใช้เวลานานมากถ้าไม่มี GPU)

In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11n.pt")      # << แก้: รุ่นใหม่/แม่นขึ้นได้ เช่น "yolo11s.pt", "yolo11m.pt" (หรือ "yolov8n.pt" รุ่นเก่า)
model.train(data="data.yaml", epochs=50, imgsz=640, batch=16)   # << แก้: ลอง epochs=10 ก่อน; GPU เล็ก ลด batch=8 หรือ imgsz=512
metrics = model.val(); print(metrics)   # ถ้าไม่มีชุด val แยก .val() อาจ error -> ก๊อปรูป train ส่วนหนึ่งไป images/val ก่อน

## ขั้นที่ 5 — ทำนาย test + สร้าง submission

รูปแบบ submission ของ detection แตกต่างกันมากในแต่ละ comp (อ่าน sample_submission ให้ดี)
ตัวอย่างล่างสร้างตาราง: id, class, confidence, x_min, y_min, x_max, y_max — แก้ให้ตรงรูปแบบจริง
ถ้าไฟล์ที่ได้ว่าง (0 แถว) แปลว่าโมเดลยังไม่เจอวัตถุ -> เทรนเพิ่มหรือลด conf

In [ ]:
import glob, pandas as pd
TEST_IMG_DIR = os.path.join(DATA_DIR, "images/test")   # << แก้: path รูป test เช่น os.path.join(DATA_DIR,"test/images")
rows = []
for p in sorted(glob.glob(os.path.join(TEST_IMG_DIR, "*"))):
    r = model.predict(p, verbose=False)[0]
    img_id = os.path.splitext(os.path.basename(p))[0]
    for b in r.boxes:
        x1,y1,x2,y2 = b.xyxy[0].tolist()
        rows.append({"id": img_id, "class": int(b.cls[0]), "confidence": float(b.conf[0]),
                     "x_min": x1, "y_min": y1, "x_max": x2, "y_max": y2})
sub = pd.DataFrame(rows)            # << แก้คอลัมน์ให้ตรง sample เช่น sub = sub.rename(columns={"class":"label"})
sub.to_csv("submission.csv", index=False)
print("บันทึก submission.csv", sub.shape); display(sub.head())

## ขั้นสุดท้าย — ส่ง submission ขึ้น Kaggle

เช็คก่อนส่งทุกครั้ง: ไฟล์ submission ต้องมีชื่อคอลัมน์ + จำนวนแถว ตรงกับ `sample_submission.csv` เป๊ะ ๆ
- บน Kaggle Notebook: กดปุ่ม `Submit` ที่แถบขวา (ง่ายสุด) หรือใช้คำสั่งข้างล่าง
- บน Colab/เครื่อง: รันเซลล์ข้างล่าง (ต้องตั้ง token แล้ว)

In [ ]:
import pandas as pd, glob, os
FILE = "submission.csv"   # << แก้เป็นไฟล์ที่จะส่ง เช่น "submission_lgbm.csv" หรือ "submission_timm.csv"
# ตรวจรูปแบบก่อนส่งอัตโนมัติ (กันแก้ config ผิดแล้วส่งฟอร์แมตเพี้ยน -> ได้ 0 คะแนน)
_sub = pd.read_csv(FILE)
_sam = glob.glob(os.path.join(DATA_DIR, "*ample*ubmission*.csv"))
if _sam:
    _s = pd.read_csv(_sam[0])
    print("คอลัมน์ตรง sample:", list(_sub.columns)==list(_s.columns), "| จำนวนแถวตรง:", len(_sub)==len(_s))
    assert list(_sub.columns)==list(_s.columns), f"คอลัมน์ไม่ตรง sample! {list(_sub.columns)} != {list(_s.columns)} -> แก้ก่อนส่ง"
print("พร้อมส่ง:", FILE, _sub.shape)
!kaggle competitions submit -c {COMP} -f {FILE} -m "yolov8 detection"
!kaggle competitions submissions -c {COMP} | head